# TrustQA on Google Colab

Multi-document question answering over PDFs that returns **answer + source document + page +
supporting quote + verification status** (Verified / Partially Supported / Insufficient Evidence).

This notebook runs the whole pipeline end to end: install, process a PDF (with OCR for scanned
pages and table extraction), ask questions, inspect citations, and optionally run the evaluation.

**Runtime:** *Runtime -> Change runtime type -> T4 GPU* (the local LLM is ~3 s/answer on a GPU,
much slower on CPU). Everything here is free: Tesseract OCR, Sentence-Transformers, FAISS and a
local `qwen2.5:3b` served by Ollama. A Groq key (free tier) is an optional stronger LLM.

## 1. Install system tools and clone the repository

In [ ]:
!apt-get -qq update && apt-get -qq install -y tesseract-ocr > /dev/null
!tesseract --version | head -1
# If the repository is private, use https://<token>@github.com/bhyreshbs/TrustQA.git
!git clone -q https://github.com/bhyreshbs/TrustQA.git
%cd TrustQA
!pip -q install -r requirements.txt && pip -q install -e . --no-deps

## 2. Choose the LLM

**Option A (default, free, no key):** run Ollama inside Colab and pull `qwen2.5:3b` (~2 GB).
**Option B:** a free Groq key from console.groq.com - set `USE_GROQ = True` below.

In [ ]:
USE_GROQ = False  # set True and paste a key to use llama-3.3-70b on Groq instead

import os, subprocess, time
if not USE_GROQ:
    !curl -fsSL https://ollama.com/install.sh | sh > /dev/null
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    !ollama pull qwen2.5:3b > /dev/null && echo "model ready"

In [ ]:
# .env for this runtime: system C (TrustQA) with the settings chosen in phases.md
env = open(".env.example").read()
if USE_GROQ:
    from getpass import getpass
    env += "\nLLM_PROVIDER=groq\nLLM_MODEL=llama-3.3-70b-versatile\nGROQ_API_KEY=" + getpass("Groq key: ")
open(".env", "w").write(env)
from trustqa.config import load_config
from trustqa.llm import check_available
cfg = load_config()
print("LLM:", check_available(cfg), "| system:", cfg.system_name)

## 3. Get PDFs

Upload your own (left sidebar -> Files, or the cell below), or download the four public pilot
papers used during development (`data/README.md` lists their sources).

In [ ]:
import pathlib, urllib.request
pdfs = pathlib.Path("data/raw_pdfs/text"); pdfs.mkdir(parents=True, exist_ok=True)
for name, url in {"rag_lewis2020.pdf": "https://arxiv.org/pdf/2005.11401",
                  "selfrag_asai2023.pdf": "https://arxiv.org/pdf/2310.11511",
                  "beir_thakur2021.pdf": "https://arxiv.org/pdf/2104.08663"}.items():
    if not (pdfs / name).exists():
        urllib.request.urlretrieve(url, pdfs / name)
# Optional: make a scanned (image-only) copy to exercise OCR
!python scripts/make_scanned_pdf.py --help | head -3
sorted(str(p) for p in pathlib.Path("data/raw_pdfs").rglob("*.pdf"))

In [ ]:
# Or upload your own PDFs
from google.colab import files  # noqa: only in Colab
uploaded = files.upload()
for name, data in uploaded.items():
    (pdfs / name).write_bytes(data)

## 4. Process the documents

`prepare_corpus` checks every file (unreadable / encrypted PDFs are reported and skipped),
OCRs scanned pages, extracts tables, chunks the text and builds the index. Results are cached by
file hash, so re-running is fast.

In [ ]:
import pandas as pd
from trustqa.corpus import prepare_corpus, system_config
docs = sorted(pathlib.Path("data/raw_pdfs").rglob("*.pdf"))
corpus = prepare_corpus(docs, system_config("C"), progress=lambda f, m: print(f"{f:4.0%} {m}") if f in (0, 1) else None)
pd.DataFrame([{"id": d.document_id, "document": d.name, "pages": d.pages, "OCR pages": len(d.ocr_pages),
               "tables": d.tables, "chunks": d.text_chunks + d.table_chunks, "error": d.error} for d in corpus.documents])

## 5. Ask questions

In [ ]:
def ask(question, c=corpus):
    a = c.ask(question)
    r = a.result
    print(f"Q: {question}\nStatus: {r.status.value}   ({a.seconds:.1f} s, {c.system})\n\n{r.answer}\n")
    for i, cit in enumerate(r.citations, 1):
        print(f"[{i}] {cit.document_name}, page {cit.page_number}  {cit.chunk_ids}")
        if cit.supporting_quote:
            print(f"    quote: {cit.supporting_quote[:300]}")
    for w in r.warnings:
        print(f"warning [{w.code.value}]: {w.message}")
    return a

ask("What nDCG@10 does BM25 achieve on TREC-COVID?")

In [ ]:
ask("Which retriever is RAG's retriever initialized from?")
ask("What was the CEO's salary in 2023?")   # not in the documents -> Insufficient Evidence

## 6. Compare with plain RAG (baseline B)

Baseline B = OCR + plain RAG: tables flattened, no verification. Its answers are never Verified.

In [ ]:
baseline = prepare_corpus(docs, system_config("B"), system="B")
for q in ["What nDCG@10 does BM25 achieve on TREC-COVID?", "What GPUs were used to train the Self-RAG models?"]:
    ask(q, corpus); ask(q, baseline); print("-" * 80)

## 7. Evaluation (optional, ~1 h on a T4)

Runs systems A, B, C and the ablations on the 38-question dev set with an LLM judge, then writes
tables to `eval/results/dev_questions/` and plots to `eval/plots/dev_questions/`. The dev
questions refer to the four pilot papers (the scanned Transformer copy is made with
`scripts/make_scanned_pdf.py`, see `data/README.md`).

In [ ]:
# !python -m eval.run_all --questions eval/dev_questions.jsonl
# print(open("eval/results/dev_questions/summary.md").read())

## 8. Web demo (optional)

The Streamlit app needs a tunnel in Colab, e.g.
`!streamlit run app/streamlit_app.py --server.headless true & npx localtunnel --port 8501`.
Locally it is just `streamlit run app/streamlit_app.py`.